# singular-matrix-mask-trick — ex1: solve a batch with some singular matrices

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `singular-matrix-mask-trick`. Running the final beacon cell reports progress against the `Numpy: Singular matrix mask trick` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Singular matrix mask trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`singular-matrix-mask-trick`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "singular-matrix-mask-trick"
DD_SUBTOPIC = "Numpy: Singular matrix mask trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Singular-matrix mask trick — quick refresher

`t.linalg.solve(A, b)` raises if **any** slice of `A` is singular. In a batched setting that's often unacceptable — a single bad slice should not kill the entire solve. The standard workaround:

1. **Detect** singular slices: `dets = t.linalg.det(A); is_singular = dets.abs() < eps`.
2. **Mask in** the identity matrix at those slices: `A[is_singular] = t.eye(n)`. Now `solve` succeeds everywhere (the identity slices return `b` unchanged).
3. **Mask out** the spurious results from the final boolean predicate: `valid = predicate & ~is_singular`.

**Why it works.** The identity overwrite is purely cosmetic — we never trust those entries, we just need solve to not crash. The mask in step 3 removes them from the answer.

**Watch out.** `A[is_singular] = t.eye(n)` relies on broadcasting: the `(n, n)` identity broadcasts to every selected slice. Confirm shapes before using; for shapes other than the last two dims you may need `A[is_singular] = t.eye(n).expand_as(A[is_singular])`.

### Exercise 1 — solve a batch with some singular matrices

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the singular-matrix-mask trick: detect singular slices via `t.linalg.det`, overwrite them with the identity, run the solve without crashing, and return per-slice `(solution, is_valid)` where `is_valid` flags non-singular slices.
> Keywords: singular, linalg, mask, boolean-index
> ```

**KCs targeted:** `singular-detect-via-det`, `singular-overwrite-identity`

Implement `ex1_solve_with_singular_mask(A, b, eps=1e-8)`.

- `A` has shape `(K, n, n)`. **Some slices may be singular.**
- `b` has shape `(K, n)`.
- `eps` is the singular-detection threshold on `|det|`.

Return `(x, is_valid)`:
- `x: (K, n)` — the solution at each slice. For singular slices this entry is undefined (we just need the solve not to crash).
- `is_valid: (K,) bool` — `True` where the original `A[k]` was non-singular.

**Algorithm.**
1. `dets = t.linalg.det(A)` then `is_singular = dets.abs() < eps`.
2. **Clone** `A` (don't mutate the input!) and overwrite singular slices with `t.eye(n)`.
3. `x = t.linalg.solve(A_safe, b)`.
4. Return `x`, `~is_singular`.

The grader explicitly constructs a batch with two known singular matrices and verifies you mask them correctly while still solving the well-conditioned ones.

In [ ]:
def ex1_solve_with_singular_mask(A: Tensor, b: Tensor, eps: float = 1e-8):
    """Returns (x, is_valid). Singular slices flagged in is_valid=False."""
    raise NotImplementedError()


def _test_ex1():
    n = 2
    # Build batch: 2 good 2x2 systems + 2 singular ones (det = 0).
    A = t.stack([
        t.tensor([[1.0, 0.0], [0.0, 1.0]]),     # identity, det=1, x=b
        t.tensor([[2.0, 1.0], [1.0, 3.0]]),     # det=5, well-conditioned
        t.tensor([[1.0, 2.0], [2.0, 4.0]]),     # SINGULAR (rows are 1:2 proportional)
        t.tensor([[0.0, 0.0], [3.0, 1.0]]),     # SINGULAR (first row all zero)
    ])
    b = t.tensor([
        [3.0, -1.0],
        [5.0,  5.0],
        [9.0, 18.0],
        [4.0,  2.0],
    ])
    A_before = A.clone()  # to confirm we didn't mutate input
    x, is_valid = ex1_solve_with_singular_mask(A, b)

    # Did NOT mutate input.
    assert t.equal(A, A_before), 'must not mutate the input A in place'

    # Shapes / dtypes.
    assert x.shape == (4, 2), f'x: expected (4,2), got {tuple(x.shape)}'
    assert is_valid.shape == (4,), f'is_valid: expected (4,), got {tuple(is_valid.shape)}'
    assert is_valid.dtype == t.bool

    # Validity mask: slices 0 and 1 valid, 2 and 3 singular.
    expected_valid = t.tensor([True, True, False, False])
    assert t.equal(is_valid, expected_valid), f'valid mask wrong: {is_valid} vs {expected_valid}'

    # Good slices must hit the right answer.
    assert t.allclose(x[0], t.tensor([3.0, -1.0]), atol=1e-5)
    assert t.allclose(x[1], t.tensor([2.0, 1.0]),  atol=1e-5)

    # Solve must finish without raising — already proven by reaching here.
    print('  singular-batch solve completed without crashing')

    # Random large batch: inject a known number of singular slices.
    rng = t.Generator().manual_seed(2)
    K = 50
    A_big = t.eye(2).expand(K, 2, 2) + 0.2 * t.randn(K, 2, 2, generator=rng)
    A_big = A_big.clone()  # break the expand-storage so we can mutate slices
    # Force slices [3, 17, 42] to be singular (rank 1).
    singular_idx = [3, 17, 42]
    for k in singular_idx:
        A_big[k] = t.tensor([[1.0, 2.0], [2.0, 4.0]])
    b_big = t.randn(K, 2, generator=rng)
    x_big, valid_big = ex1_solve_with_singular_mask(A_big, b_big)
    assert x_big.shape == (K, 2)
    # Exactly the three injected slices should be flagged invalid.
    invalid_idx = (~valid_big).nonzero(as_tuple=True)[0].tolist()
    assert sorted(invalid_idx) == sorted(singular_idx), (
        f'invalid indices mismatch: {sorted(invalid_idx)} vs {sorted(singular_idx)}'
    )
    # Reconstruction OK on valid slices.
    recon = t.einsum('kij,kj->ki', A_big[valid_big], x_big[valid_big])
    assert t.allclose(recon, b_big[valid_big], atol=1e-4), (
        'reconstruction failed on the non-singular slices'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_solve_with_singular_mask(A: Tensor, b: Tensor, eps: float = 1e-8):
    K, n, _ = A.shape
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()                # don't mutate caller's tensor
    A_safe[is_singular] = t.eye(n)    # broadcast identity into singular slices
    x = t.linalg.solve(A_safe, b)
    return x, ~is_singular
```

**Why clone.** `A[is_singular] = ...` writes through the original tensor's storage. If the caller passed a view of something larger (e.g. one part of a bigger model state), in-place mutation is a silent bug. `A.clone()` allocates fresh storage so the rewrite is scoped to our local copy.

**Why identity specifically.** Any non-singular matrix would let `solve` succeed, but the identity has the cleanest semantics — the spurious 'solution' it produces is just `b[k]` unchanged, so if you ever forget to mask it out, downstream `NaN`s won't propagate from those slices. (You'll still get *wrong* answers, but they're finite and easy to debug.)

**The det threshold.** `1e-8` is a typical default for `float32`; for `float64` use `1e-12`. A more principled approach uses the condition number (`t.linalg.cond`), but `|det| < eps` is the ARENA convention and fast.

**Caveat for n > 2.** `t.eye(n)` is `(n, n)` and broadcasts cleanly into `A[is_singular]` (which has shape `(M, n, n)` for `M` singular slices). The broadcast rule is the same one that lets `A[is_singular] = 0.0` zero a batch of matrices.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()